## Create bbox images from a video for uploading to wtb-team-classification roboflow so we can train model to detect which team the player is in:

- roboflow project: https://app.roboflow.com/btlprojects/wtb-team-classification/upload


In [11]:
import cv2
import os
import random
import numpy as np
from ultralytics import YOLO
import supervision as sv
from utils import ImageUtils
import time
from tqdm import tqdm


# SOURCE:
VIDEO_FILE_NAME = "polis-west-coast.mp4"
VIDEO_FILE_SOURCE = f"/Volumes/drive/360-camera-files/football/mixed-games/2025.02.02/flattened/{VIDEO_FILE_NAME}"
RANDOM_FRAMES_TOTAL = 10
OUTPUT_DIR = f"../../model-training/team-classification-images/{VIDEO_FILE_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# DETECTION SETUP:
PLAYERS_MODEL_PATH = os.path.join('models', 'objects', 'best-v10-1920x1920.pt')
PLAYERS_TRACKER = YOLO(PLAYERS_MODEL_PATH)
BYTE_TRACKER = sv.ByteTrack()
player_model_imgsz = PLAYERS_TRACKER.overrides.get('imgsz', (640, 640))
if isinstance(player_model_imgsz, int):              # Ensure it's a tuple (H, W)
    player_model_imgsz = (player_model_imgsz, player_model_imgsz)  # Convert to (H, W) if single int
height, width = player_model_imgsz
PLAYER_MODEL_INPUT_SIZE = {'width':width,'height':height}
print(f"-- YOLO PLAYERS_TRACKER INPUT SIZE: height={PLAYER_MODEL_INPUT_SIZE['height']}, width={PLAYER_MODEL_INPUT_SIZE['width']}")

# VIDEO CAPTURE:
CAP = cv2.VideoCapture(VIDEO_FILE_SOURCE)
ORIGINAL_VIDEO_WIDTH = int(CAP.get(cv2.CAP_PROP_FRAME_WIDTH))
ORIGINAL_VIDEO_HEIGHT = int(CAP.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"-- CAP VIDEO: height={ORIGINAL_VIDEO_HEIGHT}, width={ORIGINAL_VIDEO_WIDTH}")

def extract_random_frames_in_memory(video_path, num_frames):
    
    print('... Collecting random frames from video')

    frame_count = int(CAP.get(cv2.CAP_PROP_FRAME_COUNT))

    # Ensure the number of frames does not exceed video frame count
    num_frames = min(num_frames, frame_count)
    
    # Randomly sample frame indices
    frame_indices = sorted(random.sample(range(frame_count), num_frames))

    # save the frame to an array to process for detections:
    frames = []
    for frame_idx in frame_indices:
        CAP.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = CAP.read()
        if ret:
            frames.append(frame)
        else:
            print(f"Failed to read frame at index {frame_idx}")
    
    print(f"Frames captured: {len(frames)}")
    CAP.release()
    return frames

def yolo_detection_on_frames(frames, output_dir):
    
    print('... Processing detections')

    for frame_idx, frame in enumerate(tqdm(frames, desc="Processing Frames", unit="frame")):
        
        # resize the image and create tensor for processing:
        frame_tensor = ImageUtils.create_tensor_and_resize(frame,
                                                       PLAYER_MODEL_INPUT_SIZE['width'],
                                                       PLAYER_MODEL_INPUT_SIZE['height'],
                                                       ORIGINAL_VIDEO_WIDTH,
                                                       ORIGINAL_VIDEO_HEIGHT,
                                                       )

        crop_path = os.path.join(
                                OUTPUT_DIR,
                                f"tensor_v1-frame_{frame_idx}.jpg"
                            )
        img = (frame_tensor.cpu().numpy() * 255).astype(np.uint8)  # Scale back to 0-255 range
        #if img.shape[0] == 3:  # If in CHW format
            #img = img.transpose(1, 2, 0)  # Convert to HWC
        success = cv2.imwrite(crop_path, img)

        # Add a batch dimension
        frame_tensor = frame_tensor.unsqueeze(0)  # Shape becomes (1, 3, 640, 640)

        # YOLO object detection
        player_detections_batch = PLAYERS_TRACKER.predict(
                frame_tensor,
                conf=0.3,
                device='cpu',
                half=False,
                imgsz=(PLAYER_MODEL_INPUT_SIZE['height'],PLAYER_MODEL_INPUT_SIZE['width']),
                verbose=False
        )
        

        frame_results = []
        # loop each detection and prepare for db writing of detections:
        for idx, detection_result in enumerate(player_detections_batch):
                
                # convert to supervision
                frame_detections_supervision = sv.Detections.from_ultralytics(detection_result)
        
                # Add Tracking of Objects
                frame_detections_with_tracking = BYTE_TRACKER.update_with_detections(frame_detections_supervision)   

                # Loop results and get info from them that we want to store in db: Iterate over each detection
                
                for i in range(len(frame_detections_with_tracking.xyxy)):

                    xmin, ymin, xmax, ymax = frame_detections_with_tracking.xyxy[i].tolist()
                    confidence = float(frame_detections_with_tracking.confidence[i])
                    class_id = int(frame_detections_with_tracking.class_id[i])
                    tracker_id = int(frame_detections_with_tracking.tracker_id[i]) if frame_detections_with_tracking.tracker_id is not None else None
                    class_name = frame_detections_with_tracking.data['class_name'][i] if 'class_name' in frame_detections_with_tracking.data else 'none'

                    if class_name == 'player':
                        # scale the bounding boxes back to the original video size:
                        scaler_x = ORIGINAL_VIDEO_WIDTH/PLAYER_MODEL_INPUT_SIZE['width']
                        scaler_y = ORIGINAL_VIDEO_HEIGHT/PLAYER_MODEL_INPUT_SIZE['height']
                        # Crop the bounding box and rescale back to original xy
                        x = int(max(0, int(xmin))*scaler_x)
                        y = int(max(0, int(ymin))*scaler_y)
                        xm = int(min(frame.shape[1], int(xmax))*scaler_x)
                        ym = int(min(frame.shape[0], int(ymax))*scaler_y)

                        #print(f"bbox: {x},{y},{xm},{ym} - conf: {confidence} - class_name: {class_name}")

                        crop = frame[y:ym, x:xm]
                        if crop.size > 0:
                            #print(f"Saving crop with shape: {crop.shape}, dtype: {crop.dtype}")
                            timestamp = int(time.time() * 1000)  # Current time in milliseconds
                            crop_path = os.path.join(
                                OUTPUT_DIR,
                                f"{VIDEO_FILE_NAME}-frame_{frame_idx}_class_{class_name}_{confidence:.2f}_{timestamp}.jpg"
                            )
                            #print(f"Saving to path: {crop_path}")
                            
                            # Save the crop
                            success = cv2.imwrite(crop_path, crop)
                            '''
                            if success:
                                print(f"- Crop saved successfully: {crop_path}")
                            else:
                                print(f"Failed to save crop: {crop_path}")
                            '''

if __name__ == "__main__":

    # Extract random frames into memory
    frames = extract_random_frames_in_memory(VIDEO_FILE_SOURCE, RANDOM_FRAMES_TOTAL)

    # Perform YOLO detection and save bounding boxes
    yolo_detection_on_frames(frames, OUTPUT_DIR)

    print(f"Processing complete. Bounding boxes saved in '{OUTPUT_DIR}'.")

-- YOLO PLAYERS_TRACKER INPUT SIZE: height=1920, width=1920
-- CAP VIDEO: height=2160, width=3840
... Collecting random frames from video
Frames captured: 10
... Processing detections


Processing Frames: 100%|██████████| 10/10 [00:09<00:00,  1.10frame/s]

Processing complete. Bounding boxes saved in '../../model-training/team-classification-images/polis-west-coast.mp4'.


* TEST the predictions of the bboxes saved

In [13]:
import cv2
import os
import numpy as np
from ultralytics import YOLO
import supervision as sv
import torch
import torchvision.transforms as T
from utils import ImageUtils
import time
from tqdm import tqdm
import matplotlib.pyplot as plt
from team_assigner import TeamAssigner
from ultralytics.data.dataset import YOLODataset


TEAM_ASSIGNER = TeamAssigner([])

# DETECTION SETUP:
TEAMS_MODEL_PATH = os.path.join('models', 'teams', 'best-v4-192x192.pt')
TEAMS_TRACKER = YOLO(TEAMS_MODEL_PATH)
teams_model_imgsz = TEAMS_TRACKER.overrides.get('imgsz', (640, 640))
if isinstance(teams_model_imgsz, int):              # Ensure it's a tuple (H, W)
    teams_model_imgsz = (teams_model_imgsz, teams_model_imgsz)  # Convert to (H, W) if single int
height, width = teams_model_imgsz
TEAMS_MODEL_INPUT_SIZE = {'width':width,'height':height}
print(f"-- YOLO TEAMS_TRACKER INPUT SIZE: height={TEAMS_MODEL_INPUT_SIZE['height']}, width={TEAMS_MODEL_INPUT_SIZE['width']}")

# for each of the images in the Folder (OUTPUT_DIR) do a prediction of the classification:
VIDEO_FILE_NAME = "POLISA-AKAMAS.mp4"
OUTPUT_DIR = f"../../model-training/team-classification-images/{VIDEO_FILE_NAME}"
TENSOR_TEST = True
# Loop through all images in the folder
for image_name in os.listdir(OUTPUT_DIR):

    if image_name.lower().endswith(('.png', '.jpg', '.jpeg')):  # Filter image files
        # Load image
        image_path = os.path.join(OUTPUT_DIR, image_name)
        image = cv2.imread(image_path)
        
        ## testing predict directly from the image: without tensor convert
        if TENSOR_TEST:
            
            '''
            # Create a copy of the image and convert BGR to RGB
            img = image.copy()
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Resize to dimensions divisible by 32
            target_size = (192, 192)  # Both dimensions divisible by 32
            img = cv2.resize(img, target_size)
            
            # Convert to tensor
            img = torch.from_numpy(img).float()  # Convert to float32
            img = img.permute(2, 0, 1)  # HWC to CHW format
            img = img / 255.0  # Normalize to [0, 1]
            
            # Add batch dimension
            img = img.unsqueeze(0)
            
            print(f"\nProcessed tensor details:")
            print(f"Shape: {img.shape}")
            print(f"Dtype: {img.dtype}")
            print(f"Range: {img.min():.3f} to {img.max():.3f}")
            '''

            img = ImageUtils.create_team_assigner_tensor(image,TEAMS_MODEL_INPUT_SIZE['width'],TEAMS_MODEL_INPUT_SIZE['height'])
            results = TEAMS_TRACKER.predict(
                img,
                conf=0.3,
                device='cpu',  # Change to 'cuda' for A100
                half=False,
                imgsz=(192, 192),
                verbose=True
            )
        else:
            # Perform team detection on image
            results = TEAMS_TRACKER.predict(
                image,
                conf=0.3,
                device='cpu',
                half=False,
                imgsz=(192, 192),
                verbose=False
            )

        
        # Assign team based on detection
        #team_name, team_color = TEAM_ASSIGNER.assign_team(results)
        
        if isinstance(results, list):
            # Handle batched results: take the first result
            result = results[0]

        # Access class names from the model
        class_names = result.names if hasattr(result, 'names') else None
        
        if not class_names:
            print("Class names are not available in the result.")

        if result.probs is not None:
            # Convert probabilities to a Python dictionary
            probs_dict = {class_names[i]: float(prob) for i, prob in enumerate(result.probs.data)}
            
            # Get the most probable class and confidence
            top_class = max(probs_dict, key=probs_dict.get)
            confidence = probs_dict[top_class]
            
            #print(f"Top Classification: {top_class}, Confidence: {confidence:.2f}")
            #print("All Probabilities:", probs_dict)

        else:
            print("No classification probabilities available.")



        # Annotate image
        annotated_image = image.copy()

        # Show the image with team label
        #cv2.putText(annotated_image, f"{top_class} ({confidence*100:.2f}%)", 
        #                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        # Display the annotated image using Matplotlib
        plt.imshow(cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB))
        plt.title(f"{image_name}->{top_class}")
        plt.axis('off')  # Hide axes
        plt.show()

# using sv show the images with annotation of which team it detected:



-- YOLO TEAMS_TRACKER INPUT SIZE: height=192, width=192


FileNotFoundError: [Errno 2] No such file or directory: '../../model-training/team-classification-images/POLISA-AKAMAS.mp4'